In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
!pip install evaluate sacrebleu
!pip install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 13.2 MB/s eta 0:00:00


In [5]:
import os
import gc
import re
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from sentence_transformers import SentenceTransformer, util
import evaluate

In [7]:
class Config:
    """Configuration for ByT5 Akkadian-to-English Machine Translation"""

    # ====== Model Configuration ======
    MODEL_NAME = "google/byt5-base"
    PRETRAINED_DIR = (
        None  # Set to a path to resume training; None to start from scratch
    )
    PRETRAINED_DIR = "/content/drive/MyDrive/kaggle/Akkadian/models/byt5-akkadian-model"
    MAX_LENGTH = 512

    # ====== Training Configuration ======
    INPUT_DIR = "/content/drive/MyDrive/kaggle/Akkadian/data"
    SEED = 42
    EPOCHS = 1
    LEARNING_RATE = 1e-4
    TRAIN_BATCH_SIZE = 2  # per_device_train_batch_size
    EVAL_BATCH_SIZE = 2  # per_device_eval_batch_size
    GRADIENT_ACCUMULATION_STEPS = 8
    OUTPUT_DIR = "/content/drive/MyDrive/kaggle/Akkadian/models/byt5-akkadian-model"

In [8]:
import wandb

wandb.login()


RUN_NAME = "ByT5-deeppast-v1"
wandb.init(
    project="deep-past-akkadian",
    name=RUN_NAME,
    config={
        "model": Config.MODEL_NAME,
        "seed": Config.SEED,
        "max_length": Config.MAX_LENGTH,
        "epochs": Config.EPOCHS,
        "learning_rate": Config.LEARNING_RATE,
        "train_batch_size": Config.TRAIN_BATCH_SIZE,
        "eval_batch_size": Config.EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": Config.GRADIENT_ACCUMULATION_STEPS,
    },
)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yuzurihainori315 (yuzurihainori315-national-institute-of-technology-ibarak) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [9]:
# Fix the seed (for reproducibility).
def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)


seed_everything(Config.SEED)

In [10]:
INPUT_DIR = Config.INPUT_DIR
train_df = pd.read_csv(f"{INPUT_DIR}/train.csv")
test_df = pd.read_csv(f"{INPUT_DIR}/test.csv")

In [11]:
print(f"Original Train Data: {len(train_df)} docs")

Original Train Data: 1561 docs


In [12]:
def simple_sentence_aligner(df):
    """
    【戦略の肝】
    Trainデータの「文書(複数文)」を、Testデータと同じ「文(1文)」に分割します。
    ここでは「英語の文数」と「アッカド語の行数」が一致する場合のみ分割する
    というヒューリスティック（簡易ルール）を使います。
    """
    aligned_data = []

    for idx, row in df.iterrows():
        src = str(row["transliteration"])
        tgt = str(row["translation"])

        # Split the English text by sentence-ending punctuation.
        tgt_sents = [t.strip() for t in re.split(r"(?<=[.!?])\s+", tgt) if t.strip()]

        # Assume the Akkadian text is often separated by newlines and split accordingly.
        src_lines = [s.strip() for s in src.split("\n") if s.strip()]

        # If the counts match, trust it as 1-to-1 pairs and use the split version.
        if len(tgt_sents) > 1 and len(tgt_sents) == len(src_lines):
            for s, t in zip(src_lines, tgt_sents):
                if len(s) > 3 and len(t) > 3:  # Remove junk/noisy data.
                    aligned_data.append({"transliteration": s, "translation": t})
        else:
            # If splitting fails (counts don't match), keep the original document pair as-is (safe fallback).
            aligned_data.append({"transliteration": src, "translation": tgt})

    return pd.DataFrame(aligned_data)

In [13]:
# Run data augmentation.
train_expanded = simple_sentence_aligner(train_df)
print(f"Expanded Train Data: {len(train_expanded)} sentences (Alignment applied)")

# Convert to Hugging Face Dataset format & split into Train/Val.
dataset = Dataset.from_pandas(train_expanded)
# Create a validation set with test_size=0.1.
split_datasets = dataset.train_test_split(test_size=0.1, seed=42)
# After splitting, the keys are 'train' and 'test' (we'll use 'test' as validation).

Expanded Train Data: 1561 sentences (Alignment applied)


In [14]:
def create_bidirectional_data(dataset_split):
    df = dataset_split.to_pandas()

    # 方向1: Akkadian -> English (元のタスク)
    df_fwd = df.copy()
    df_fwd["input_text"] = "translate Akkadian to English: " + df_fwd[
        "transliteration"
    ].astype(str)
    df_fwd["target_text"] = df_fwd["translation"].astype(str)

    # 方向2: English -> Akkadian (逆翻訳タスク)
    df_bwd = df.copy()
    df_bwd["input_text"] = "translate English to Akkadian: " + df_bwd[
        "translation"
    ].astype(str)
    df_bwd["target_text"] = df_bwd["transliteration"].astype(str)

    # 結合してシャッフル
    df_combined = pd.concat([df_fwd, df_bwd], ignore_index=True)
    df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

    return Dataset.from_pandas(df_combined)


def create_unidirectional_data(dataset_split):
    # Validation用: 形式だけ揃える (Akkadian -> Englishのみ)
    df = dataset_split.to_pandas()
    df["input_text"] = "translate Akkadian to English: " + df["transliteration"].astype(
        str
    )
    df["target_text"] = df["translation"].astype(str)
    return Dataset.from_pandas(df)


# Trainデータは双方向化 (データ数が2倍になります)
bidirectional_train = create_bidirectional_data(split_datasets["train"])

# Validationデータは単方向のまま (評価のため)
unidirectional_val = create_unidirectional_data(split_datasets["test"])

print(f"Train samples: {len(bidirectional_train)} (Bidirectional)")
print(f"Val samples:   {len(unidirectional_val)} (Unidirectional)")

Train samples: 2808 (Bidirectional)
Val samples:   157 (Unidirectional)


In [15]:
# ==========================================
# 3. Tokenization & preprocessing
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)


def preprocess_function(examples):
    # データ作成時にPrefix込みの "input_text" と "target_text" を作っているのでそれを使う
    inputs = [str(ex) for ex in examples["input_text"]]
    targets = [str(ex) for ex in examples["target_text"]]

    model_inputs = tokenizer(inputs, max_length=Config.MAX_LENGTH, truncation=True)
    labels = tokenizer(targets, max_length=Config.MAX_LENGTH, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


# 作成した新しいデータセットに対してmapを適用
tokenized_train = bidirectional_train.map(preprocess_function, batched=True)
tokenized_val = unidirectional_val.map(preprocess_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2808 [00:00<?, ? examples/s]

Map:   0%|          | 0/157 [00:00<?, ? examples/s]

In [16]:
from transformers import TrainerCallback

DEBUG_EVAL_PRINTED = True


class PrintMetricsCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        epoch = getattr(state, "epoch", None)
        # find the most recent training loss from the log history
        train_loss = None
        for entry in reversed(state.log_history):
            if "loss" in entry:
                train_loss = entry["loss"]
                break
        prefix = f"[Epoch {epoch}] " if epoch is not None else ""
        # if train_loss is not None:
        #     print(prefix + f"Train Loss={train_loss:.4f}")
        if metrics:
            keys = ["eval_loss", "eval_chrf", "eval_bleu", "eval_geo_mean"]
            metric_items = ", ".join(
                f"{k}={metrics[k]:.4f}" for k in keys if k in metrics
            )
            print(prefix + f"Train Loss={train_loss:.4f}, " + metric_items)

In [ ]:
# ==========================================
# 4. Model training (fine-tuning)
# ==========================================
gc.collect()
torch.cuda.empty_cache()

# Load model based on PRETRAINED_DIR
if Config.PRETRAINED_DIR is not None:
    print(f"Loading pretrained model from {Config.PRETRAINED_DIR}...")
    tokenizer = AutoTokenizer.from_pretrained(
        Config.PRETRAINED_DIR,
        local_files_only=True,
        use_fast=False,  # byt5 uses a byte-level tokenizer; safe to set use_fast=False
    )
    model = AutoModelForSeq2SeqLM.from_pretrained(
        Config.PRETRAINED_DIR,
        local_files_only=True,
    )
    print("✓ Pretrained model loaded successfully")
else:
    print(f"Training from scratch using {Config.MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(Config.MODEL_NAME)
    print("✓ Base model loaded successfully")

Loading pretrained model from /content/drive/MyDrive/kaggle/Akkadian/models/byt5-akkadian-model...


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

In [29]:
# --- Memory Cleanup before Training ---
import gc
import torch

# sentence_transformers model is not needed during training
# Force release all unreferenced GPU memory
gc.collect()
torch.cuda.empty_cache()

# Check memory status
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU memory reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

GPU memory allocated: 8.23 GB
GPU memory reserved:  8.34 GB


In [31]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 評価指標の準備
metric_chrf = evaluate.load("chrf")
metric_bleu = evaluate.load("sacrebleu")


def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Seq2SeqTrainer/Trainer の返り値が tuple になることがあるので保険
    if isinstance(preds, tuple):
        preds = preds[0]

    # ★ここが重要★
    # preds が logits の場合: (batch, seq, vocab) になりがち → argmax で token ids に変換
    if hasattr(preds, "ndim") and preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    # int 化 & 範囲外対策（ByT5 の decode が chr を使うため）
    preds = preds.astype(np.int64)
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)
    preds = np.clip(preds, 0, tokenizer.vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # === Fix: Replace -100 in labels before decoding ===
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # ===================================================

    # Wrap references in list of lists for both metrics to be safe
    formatted_refs = [[x] for x in decoded_labels]

    chrf = metric_chrf.compute(predictions=decoded_preds, references=formatted_refs)[
        "score"
    ]
    bleu = metric_bleu.compute(predictions=decoded_preds, references=formatted_refs)[
        "score"
    ]
    geo_mean = (chrf * bleu) ** 0.5 if chrf > 0 and bleu > 0 else 0.0

    return {"chrf": chrf, "bleu": bleu, "geo_mean": geo_mean}


args = Seq2SeqTrainingArguments(
    output_dir=Config.OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=Config.LEARNING_RATE,
    optim="adafactor",
    label_smoothing_factor=0.2,
    # === Key fixes ===
    fp16=False,  # ★Set to False to prevent a NaN error (required).
    bf16=True,  # ★Set to True if your GPU supports it (e.g., A100); otherwise False.
    per_device_train_batch_size=Config.TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=Config.EVAL_BATCH_SIZE,
    gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION_STEPS,
    # ======================
    generation_max_length=Config.MAX_LENGTH,  # <--- Add this (512). Crucial for ByT5.
    generation_num_beams=2,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=Config.EPOCHS,
    predict_with_generate=True,
    logging_steps=10,  # Inspect logs in more detail.
    report_to="none",
    # show/use competition_metric as the main criterion
    load_best_model_at_end=True,
    metric_for_best_model="geo_mean",
    greater_is_better=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[PrintMetricsCallback()],
)

print("Starting Training (FP32 mode)...")
trainer.train()


Starting Training (FP32 mode)...


OutOfMemoryError: CUDA out of memory. Tried to allocate 12.00 MiB. GPU 0 has a total capacity of 39.49 GiB of which 2.62 MiB is free. Process 2718 has 31.20 GiB memory in use. Including non-PyTorch memory, this process has 8.27 GiB memory in use. Of the allocated memory 7.57 GiB is allocated by PyTorch, and 199.41 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# --- Save Model ---
# Important: the model saved here will be loaded in the next notebook.
trainer.save_model(Config.OUTPUT_DIR)
tokenizer.save_pretrained(Config.OUTPUT_DIR)
print(f"Model saved to {Config.OUTPUT_DIR}")